 SDXL (Fooocus)
Fooocus 2.5.5 with your checkpoints and LoRAs, at http://sdxl:7865 on Tailscale.

1. **Runtime > Change runtime type > T4 GPU**
2. **Secrets** (key icon, left), allow notebook access: `HF_TOKEN` (gated or private repos),
   `CIVITAI_TOKEN` (Civitai downloads), `TS_AUTHKEY` (Tailscale, tick Reusable + Ephemeral; only the app notebooks).
3. Run **Setup** (re-runs skip finished steps), then the cells below in order.

In [ ]:
#@title Setup: repo, venv, secrets, Drive
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/kopeng-mi/SDXL_Research.git"  #@param {type:"string"}
BRANCH = "step5-notebooks"  #@param {type:"string"}
USE_DRIVE = False  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/SDXLR"  #@param {type:"string"}
GROUPS = None  # venv install groups for this notebook (None = no shared venv)
REPO = Path("/content/SDXL_Research")

if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--recurse-submodules", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    # follow BRANCH even if an earlier run cloned another one
    subprocess.run(["git", "fetch", "-q", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO, check=True)
    subprocess.run(["git", "submodule", "update", "-q", "--init", "--recursive"], cwd=REPO, check=True)

# The kernel only orchestrates (stdlib); GPU work runs in venv subprocesses,
# so this kernel never holds VRAM.
sys.path.insert(0, str(REPO / "src"))
from sdxlr import nb
if GROUPS is not None:
    r = subprocess.run([sys.executable, "-m", "sdxlr.bootstrap", "--groups", GROUPS], stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, text=True, env={**os.environ, "PYTHONPATH": str(REPO / "src")})
    print(r.stdout[-3000:])
    assert r.returncode == 0, "setup failed (see above)"
found = nb.load_secrets()
print("secrets:", ", ".join(f"{k} {'found' if v else 'missing'}" for k, v in found.items()))
ROOT = nb.drive_mount(DRIVE_ROOT) if USE_DRIVE else None
print("drive:", ROOT or "off")

# 1. Install Fooocus
Fooocus 2.5.5 in its own Python 3.11 venv (about 4 min the first time; re-runs skip it).

In [ ]:
#@title Install Fooocus
from sdxlr import fooocus
fooocus.setup()

# 2. Models
Pick a base or paste any SDXL checkpoint: an HF file `org/repo/file.safetensors`, a Civitai or https link,
or a Drive path. Checkpoints in `SDXLR/models/sdxl` and LoRAs in `SDXLR/loras/sdxl` (your trained ones)
show up in Fooocus by themselves when Drive is on.

In [ ]:
#@title Models
MODEL = "sdxl-base"  #@param ["sdxl-base", "pony-v6", "illustrious-0.1", "noobai-eps-1.1"]
CUSTOM_MODEL = ""  #@param {type:"string"}
EXTRA_LORAS = ""  #@param {type:"string"}
#@markdown EXTRA_LORAS: comma list of LoRA links or .zip files (HF, Civitai, https, Drive).
from sdxlr import fetch, loraformat, paths
src = CUSTOM_MODEL.strip() or paths.SDXL_BASES[MODEL]
ckpt = fetch.fetch(src, fooocus.CKPTS)
loraformat.check_sdxl_checkpoint(ckpt)
if loraformat.is_vpred(ckpt):
    raise SystemExit(f"{ckpt.name} is a v-prediction model; Fooocus 2.5.5 cannot run those (the images come out grey). "
                     "Use an eps model here, or the train-test-sdxl notebook to test v-pred LoRAs.")
for s in [x.strip() for x in EXTRA_LORAS.split(",") if x.strip()]:
    print("LoRAs:", [p.name for p in fetch.fetch_loras(s)])
CKPT_DIRS = list(dict.fromkeys([fooocus.CKPTS, ckpt.parent] + ([ROOT / "models" / "sdxl"] if ROOT else [])))
LORA_DIRS = [fooocus.LORAS, paths.LORAS] + ([ROOT / "loras" / "sdxl"] if ROOT else [])
print(f"default model: {ckpt.name}")
print(f"checkpoints: {len(fooocus.checkpoints(CKPT_DIRS))}, LoRAs: {len(fooocus.checkpoints(LORA_DIRS))}")

# 3. Launch
Open **http://sdxl:7865** on a device signed in to your Tailscale (needs TS_AUTHKEY in Secrets).
Without a key it falls back to a public gradio.live link (anyone with the link can use it).
The cell keeps running while Fooocus is open; stop it to close Fooocus.

In [ ]:
#@title Launch Fooocus
USE_TAILSCALE = True  #@param {type:"boolean"}
OUTPUTS_TO_DRIVE = False  #@param {type:"boolean"}
EXTRA_ARGS = ""  #@param {type:"string"}
import os
from sdxlr import tailnet
out = ROOT / "outputs" / "fooocus" if ROOT and OUTPUTS_TO_DRIVE else fooocus.OUTPUTS
fooocus.write_config(ckpt.name, CKPT_DIRS, LORA_DIRS, out)
share = True
if USE_TAILSCALE and os.environ.get("TS_AUTHKEY"):
    print("open", tailnet.up(fooocus.PORT, hostname="sdxl"))
    share = False
elif USE_TAILSCALE:
    print("TS_AUTHKEY is missing: using a public gradio.live link instead")
os.environ["MPLBACKEND"] = "Agg"
nb.serve(fooocus.launch_args(share=share, extra=EXTRA_ARGS), cwd=fooocus.DIR, py=str(fooocus.PY))

# 4. Finish
Deletes the pictures on the VM (the Drive copy stays when OUTPUTS_TO_DRIVE was on), signs out of Tailscale,
and saves Drive. Then **Runtime > Disconnect and delete runtime**.

In [ ]:
#@title Wipe outputs and finish
print(f"deleted {fooocus.wipe_outputs()} files")
tailnet.down()
if ROOT:
    from sdxlr import drive
    drive.flush_and_unmount()